# Nowak Oak Creek Example

Notebook conversion of `example_scripts/nowak_oakcreek_example.py`.

This example uses real Oak Creek monthly and simulated annual flow data from `data/oak_creek_monthly_af.xlsx` and writes simulated monthly traces to `output/oak_sim_monthly.xlsx`.

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd

from borgRWhelper.nowak import sim_multi_trace

In [3]:
def main():
    input_path = Path("data/oak_creek_monthly_af.xlsx")
    output_dir = Path("output")
    output_dir.mkdir(parents=True, exist_ok=True)
    output_path = output_dir / "oak_sim_monthly.xlsx"

    oak_obs_monthly_df = pd.read_excel(input_path,
                                       sheet_name="Monthly Data",
                                       index_col=0)

    oak_sim_ann_df = pd.read_excel(input_path,
                                   sheet_name="Simulated Annual",
                                   index_col=0)

    # convert to np array with shape as below
    # shape: sequences in rows; years in columns
    oak_sim_ann_flow = oak_sim_ann_df.to_numpy(copy=True)

    oak_obs_years = oak_obs_monthly_df.Year.unique()
    month_labels = oak_obs_monthly_df.loc[oak_obs_monthly_df["Year"] == oak_obs_years[0], "Month"].to_numpy()

    oak_obs_monthly = oak_obs_monthly_df["Volume (af)"].to_numpy()

    # shape: years in rows, monthly values in columns
    oak_obs_monthly_mat = oak_obs_monthly.reshape((len(oak_obs_years), 12))

    oak_obs_ann_flow = np.sum(oak_obs_monthly_mat, axis=1)

    # Convert annual totals to a column vector with shape (num_years, 1) so each
    # row of monthly values is divided by that same year's annual total.
    oak_obs_ann_flow_col = oak_obs_ann_flow.reshape(-1, 1)

    # shape: years in rows, monthly proportions in columns
    oak_obs_p = oak_obs_monthly_mat / oak_obs_ann_flow_col

    # reset the random number generator to produce consistent results for this example
    oak_rng = np.random.default_rng(seed=42)

    # sim_multi_trace returns one column per simulated trace/replicate and stacks
    # all within-year monthly values down the rows.
    oak_sim_monthly = sim_multi_trace(oak_rng, oak_obs_ann_flow,
                                      oak_obs_p, oak_obs_years,
                                      oak_sim_ann_flow,
                                      repl=10, print_results=False)

    # The simulated annual input has 5 years, so the output rows are grouped into
    # 5 simulated years x 12 months. Build a MultiIndex so those stacked rows are
    # labeled as (SimYear, Month) instead of plain integer row numbers.
    sim_year_labels = [f"SimYear_{i}" for i in range(1, oak_sim_ann_flow.shape[1] + 1)]
    row_index = pd.MultiIndex.from_product([sim_year_labels, month_labels], names=["SimYear", "Month"])

    # With repl=10 and 5 input annual sequences, sim_multi_trace generates 50 output
    # traces total. Name the columns explicitly so the Excel output is easier to inspect.
    column_labels = [f"Trace_{i:02d}" for i in range(1, oak_sim_monthly.shape[1] + 1)]

    oak_sim_monthly_df = pd.DataFrame(oak_sim_monthly, index=row_index, columns=column_labels)
    print(oak_sim_monthly_df.head(12).round(2))

    oak_sim_monthly_df.to_excel(output_path)
    print(f"Wrote: {output_path}")
    return oak_sim_monthly_df

In [4]:
oak_sim_monthly_df = main()
oak_sim_monthly_df.head()

                 Trace_01  Trace_02  Trace_03  Trace_04  Trace_05  Trace_06  \
SimYear   Month                                                               
SimYear_1 1       1156.91   2041.16   1079.42  33076.06   3591.56   1251.28   
          2       1145.79   3024.03   1049.78   4728.32   5016.91   1193.61   
          3       1150.84   9741.41    996.96  22285.47   8499.89   1303.94   
          4        987.02   2055.27    892.26  10346.61   2336.95    939.09   
          5        771.61   1464.22    643.95   1392.46   1727.78    710.90   
          6        580.48   1089.72    575.38    996.92   1212.15    539.13   
          7        578.46   1595.19    691.20   1362.19   1111.09    598.06   
          8       1225.68   2600.04    826.48   1616.47   1479.83    662.00   
          9        880.83   1370.60    579.09   1574.09   1948.87   1396.72   
          10       888.92   3009.38    701.39   1802.13   1958.07    857.59   
          11      1218.60  11026.25    918.21   2675

Trace_01     Trace_02     Trace_03      Trace_04  \
SimYear   Month                                                        
SimYear_1 1      1156.912142  2041.156537  1079.424905  33076.056335   
          2      1145.787987  3024.034873  1049.775466   4728.322147   
          3      1150.844421  9741.412425   996.962402  22285.472246   
          4       987.015954  2055.271677   892.262818  10346.610179   
          5       771.611857  1464.222498   643.948763   1392.463629   

                    Trace_05     Trace_06      Trace_07     Trace_08  \
SimYear   Month                                                        
SimYear_1 1      3591.556975  1251.280060  15910.378419  1069.345291   
          2      5016.909189  1193.605829  21381.944212  1020.056831   
          3      8499.892945  1303.939141  10273.102558  1114.347798   
          4      2336.953992   939.086939   1961.056073   802.544712   
          5      1727.779344   710.897589   1446.783449   607.533848   

                     Trace_09     Trace_10  ...     Trace_41      Trace_42  \
SimYear   Month                             ...                              
SimYear_1 1       3652.967369  2717.871356  ...  1156.912142  16264.416014   
          2       4821.424944  2275.476408  ...  1145.787987  11708.777183   
          3      31683.649635  1915.633627  ...  1150.844421  19730.971821   
          4      25320.680661  1648.926625  ...   987.015954   2152.512708   
          5       1750.636438  1325.068122  ...   771.611857   1596.229858   

                    Trace_43      Trace_44     Trace_45     Trace_46  \
SimYear   Month                                                        
SimYear_1 1      1050.518939   4833.106248  2160.438039   990.497486   
          2      1405.685996  18179.625720  4176.064107   939.501576   
          3      2194.456508  24475.827946  3958.846153  2593.926584   
          4       698.877113   4359.198554  1735.786721  1169.963862   
          5       523.496854   2572.641769  1491.171907   647.255783   

                     Trace_47    Trace_48      Trace_49     Trace_50  
SimYear   Month                                                       
SimYear_1 1      16264.416014  988.698367   9451.289850  4318.621812  
          2      11708.777183  979.191652   8163.995212  2552.416771  
          3      19730.971821  983.512886  33201.810279  4727.503075  
          4       2152.512708  843.504900  13231.017995  2444.333976  
          5       1596.229858  659.420327   1721.707342  1710.351604  

[5 rows x 50 columns]